# MacroRegimes3 — Part 4: Gaussian Hidden Markov Model Regime Classifier

A multivariate Gaussian HMM is fitted to the sign-corrected proxy variables from `PVRFClassifier.ipynb`. Unlike the
tree's hard greedy splits, the HMM learns regime persistence and transition dynamics jointly and returns posterior
probabilities; Viterbi recovers the globally most likely state path. Split out of `MacroRegimes2.ipynb`, final section.

**Changes vs MacroRegimes2 (fixes):**
1. **Covariance**: `diag` by default instead of `full`. With 12 proxy variables and 3 states, a full covariance has 278 free
   parameters for 235 observations and the 53-month state fitted a 78-parameter covariance from 53 rows — which is why a third
   of the restarts threw "not converging" warnings. `cov_type` is still a switch.
2. **Number of states** is checked with BIC over 2–5 states instead of being inherited from the tree.
3. **Restart identifiability**: EM is non-convex; besides picking the max-likelihood restart we check whether the best restarts
   agree on the state path (Adjusted Rand Index).
4. **Filtered probabilities**: `predict_proba` returns forward-backward *smoothed* posteriors, which use future data.
   Real-time (forward-only, *filtered*) probabilities are computed alongside and saved; the old plot title said "Viterbi" but showed smoothed posteriors.
5. **Agreement with the tree** is measured with permutation-invariant ARI/NMI as well as canonical-label accuracy.
6. A **macro-only HMM** (proxy variables built without bond-market variables) tests whether the regimes survive without yield information.

**Inputs:** `MacroRegimes3_PVRF_ProxyVariables.csv`, `_ProxyVariables_xbond.csv` (PV-RF), `MacroRegimes3_Tree_Labels.csv` (Tree),
`MacroRegimes3_Data_Target.csv`, `_Levels.csv`, `_Recession.csv`, `_Columns.csv` (Data).

**Outputs:** `MacroRegimes3_HMM_TimeSeries.csv` (Viterbi regime, smoothed and filtered probabilities), `_HMM_TransitionMatrix.csv`,
`_HMM_EmissionMeans.csv`, `_HMM_RegimeSummary.csv`, `_HMM_ModelStats.csv`, `_HMM_ModelSelection.csv`, `_HMM_xbond_TimeSeries.csv`.

## 0. Imports and User Inputs

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, confusion_matrix

try:
    from hmmlearn.hmm import GaussianHMM
except ImportError:
    !pip install hmmlearn -q
    from hmmlearn.hmm import GaussianHMM

# Folder holding every CSV input and output (locally: DataAndResults/ next to the MacroRegimes3/ notebook folder)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    data_folder = '/content/drive/MyDrive/DataAndResults'
except ImportError:
    data_folder = os.path.join('..', 'DataAndResults')

In [ ]:
# HMM hyperparameters
n_states = 3          # number of hidden regimes (kept at 3 for comparability with the tree; see the BIC table in Section 2)
cov_type = 'diag'     # 'diag' (default, fix 1) or 'full' (as in MacroRegimes2)
min_covar = 1e-3      # floor added to the variances for numerical stability
n_iter = 1000         # EM iterations per restart
n_restarts = 50       # random restarts (EM is non-convex)
n_restarts_selection = 10  # restarts per candidate n_states in the BIC comparison
candidate_states = [2, 3, 4, 5]

regime_colors = {0: 'coral', 1: 'steelblue', 2: 'mediumseagreen', 3: 'orchid', 4: 'goldenrod'}

In [ ]:
# Load inputs
data_pv       = pd.read_csv(f'{data_folder}/MacroRegimes3_PVRF_ProxyVariables.csv', index_col='date', parse_dates=True)
data_pv_xbond = pd.read_csv(f'{data_folder}/MacroRegimes3_PVRF_ProxyVariables_xbond.csv', index_col='date', parse_dates=True)
tree_labels   = pd.read_csv(f'{data_folder}/MacroRegimes3_Tree_Labels.csv', index_col='date', parse_dates=True)
target        = pd.read_csv(f'{data_folder}/MacroRegimes3_Data_Target.csv', index_col='date', parse_dates=True)
data          = pd.read_csv(f'{data_folder}/MacroRegimes3_Data_Levels.csv', index_col='date', parse_dates=True)
recession     = pd.read_csv(f'{data_folder}/MacroRegimes3_Data_Recession.csv', index_col='date', parse_dates=True)
columns       = pd.read_csv(f'{data_folder}/MacroRegimes3_Data_Columns.csv')

common_idx = data_pv.index
all_cols = columns['variable'].tolist()
yields_svensson = target[[c for c in target.columns if c.startswith('Y')]]
regime_labels_canonical = tree_labels['regime_canonical'].values
recession_aligned = recession.reindex(common_idx, method='ffill').fillna(0)
rec_mask = recession_aligned['USREC'].values == 1

X_hmm = data_pv.values
pv_cols = data_pv.columns.tolist()
print(f'HMM input: {X_hmm.shape[0]} months × {X_hmm.shape[1]} proxy variables: {pv_cols}')

## 1. Helper Functions

In [ ]:
def fit_hmm(X, n_states, n_restarts, cov_type=cov_type, verbose=False):
    """Fit a GaussianHMM with several random restarts. Returns (best model, list of (log-likelihood, model))."""
    fits = []
    for seed in range(n_restarts):
        try:
            model = GaussianHMM(n_components=n_states, covariance_type=cov_type, min_covar=min_covar,
                                n_iter=n_iter, tol=1e-6, random_state=seed)
            model.fit(X)
            score = model.score(X)
            fits.append((score, model))
            if verbose:
                print(f'  restart {seed:3d}: log-likelihood = {score:,.2f}' + ('  NEW BEST' if score == max(f[0] for f in fits) else ''))
        except Exception as e:
            if verbose:
                print(f'  restart {seed:3d}: failed ({e})')
    fits.sort(key=lambda f: f[0], reverse=True)
    return fits[0][1], fits

def n_parameters(n_states, n_features, cov_type):
    """Free parameters: initial distribution + transition matrix + means + covariances."""
    cov = n_states * n_features * (n_features + 1) // 2 if cov_type == 'full' else n_states * n_features
    return (n_states - 1) + n_states * (n_states - 1) + n_states * n_features + cov

def information_criteria(model, X, cov_type):
    log_lik = model.score(X)
    k = n_parameters(model.n_components, X.shape[1], cov_type)
    return log_lik, k, -2 * log_lik + 2 * k, -2 * log_lik + k * np.log(len(X))

def filtered_probabilities(model, X):
    """Real-time posteriors P(state_t | x_1..x_t): run the forward-backward pass on each prefix and keep its last row."""
    probs = np.zeros((len(X), model.n_components))
    for t in range(len(X)):
        probs[t] = model.predict_proba(X[:t + 1])[-1]
    return probs

def sort_by_spread(labels, index, yields_df):
    """Canonical relabelling by mean 10Y-2Y spread (0 = flattest, 2 = steepest), consistent with the tree. Returns (labels, order)."""
    spread = (yields_df.loc[index, 'Y120M'] - yields_df.loc[index, 'Y24M']).values
    mean_spread = {s: spread[labels == s].mean() for s in np.unique(labels)}
    order = sorted(mean_spread, key=mean_spread.get)   # order[new] = old
    mapping = {old: new for new, old in enumerate(order)}
    return np.vectorize(mapping.get)(labels), order

def decode(model, X, index):
    """Viterbi labels, smoothed and filtered posteriors, all in canonical (spread-ordered) labels."""
    raw = model.predict(X)
    labels, order = sort_by_spread(raw, index, yields_svensson)
    smoothed = model.predict_proba(X)[:, order]
    filtered = filtered_probabilities(model, X)[:, order]
    return labels, smoothed, filtered, order

In [ ]:
def shade_regimes(ax, dates, labels, colors, alpha=0.25):
    labels = np.asarray(labels)
    for label in np.unique(labels):
        in_block, block_start = False, None
        for i, date in enumerate(dates):
            if labels[i] == label and not in_block:
                in_block, block_start = True, date
            elif labels[i] != label and in_block:
                in_block = False
                ax.axvspan(block_start, date, alpha=alpha, color=colors[label], lw=0, zorder=0)
        if in_block:
            ax.axvspan(block_start, dates[-1], alpha=alpha, color=colors[label], lw=0, zorder=0)

def shade_recessions(ax, recession, alpha=0.35):
    in_rec, rec_start = False, None
    for date, val in recession['USREC'].items():
        if val == 1 and not in_rec:
            in_rec, rec_start = True, date
        elif val == 0 and in_rec:
            in_rec = False
            ax.axvspan(rec_start, date, alpha=alpha, color='grey', lw=0, zorder=1)
    if in_rec:
        ax.axvspan(rec_start, recession.index[-1], alpha=alpha, color='grey', lw=0, zorder=1)

def add_regime_legend(ax, labels, colors, loc='upper right'):
    for label in np.unique(labels):
        ax.scatter([], [], color=colors[label], alpha=0.5, label=f'Regime {label}', s=100, marker='s')
    ax.scatter([], [], color='grey', alpha=0.5, label='NBER Recession', s=100, marker='s')
    ax.legend(loc=loc, ncol=len(np.unique(labels)) + 1)

## 2. How Many States? (BIC)
Lower is better. Three states are kept below for comparability with the tree; change `n_states` if you prefer the BIC choice.

In [ ]:
selection_rows = []
for k in candidate_states:
    model_k, _ = fit_hmm(X_hmm, k, n_restarts_selection)
    log_lik, n_par, aic, bic = information_criteria(model_k, X_hmm, cov_type)
    selection_rows.append({'n_states': k, 'log_likelihood': log_lik, 'n_params': n_par, 'AIC': aic, 'BIC': bic})
model_selection = pd.DataFrame(selection_rows).set_index('n_states')
print(model_selection.round(2).to_string())
print(f'\nBIC prefers {model_selection["BIC"].idxmin()} states; AIC prefers {model_selection["AIC"].idxmin()}. Using n_states = {n_states}.')

## 3. Fit With Restarts and Check Identifiability

In [ ]:
print(f'Fitting GaussianHMM ({n_states} states, {cov_type} covariance) with {n_restarts} restarts...')
best_model, fits = fit_hmm(X_hmm, n_states, n_restarts, verbose=True)
best_score = fits[0][0]
print(f'\nBest log-likelihood: {best_score:,.2f} | converged: {best_model.monitor_.converged}')

plt.figure(figsize=(8, 3))
plt.hist([f[0] for f in fits], bins=20, edgecolor='white')
plt.axvline(best_score, color='red', ls='--', label=f'Best: {best_score:,.1f}')
plt.xlabel('Log-likelihood'); plt.ylabel('Restarts'); plt.title('Log-likelihood across restarts'); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Do the best restarts agree on the state path? (ARI = 1 means identical partitions up to relabelling)
top_k = 5
top_paths = [f[1].predict(X_hmm) for f in fits[:top_k]]
ari = np.array([[adjusted_rand_score(a, b) for b in top_paths] for a in top_paths])
ari_df = pd.DataFrame(ari, index=[f'#{i+1} ({fits[i][0]:,.1f})' for i in range(top_k)], columns=[f'#{i+1}' for i in range(top_k)])
print(f'Adjusted Rand Index between the Viterbi paths of the top {top_k} restarts (rows labelled with log-likelihood):')
print(ari_df.round(3).to_string())
print(f'\nMean off-diagonal ARI: {ari[~np.eye(top_k, dtype=bool)].mean():.3f}  (close to 1 = the regimes are well identified)')

## 4. Decoding: Viterbi Path, Smoothed and Filtered Probabilities

In [ ]:
hmm_labels, hmm_smoothed, hmm_filtered, state_order = decode(best_model, X_hmm, common_idx)
hmm_labels_filtered = hmm_filtered.argmax(axis=1)  # what a real-time user would have called each month

print('Canonical ordering by mean 10Y-2Y spread:')
for new, old in enumerate(state_order):
    print(f'  raw state {old} → Regime {new}: {(hmm_labels == new).sum()} months ({(hmm_labels == new).mean():.1%})')
print(f'\nReal-time (filtered argmax) vs Viterbi labels agree on {(hmm_labels_filtered == hmm_labels).mean():.1%} of months')

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for ax, probs, title in [(axes[0], hmm_smoothed, 'Smoothed posteriors (forward-backward, uses future data)'),
                         (axes[1], hmm_filtered, 'Filtered posteriors (forward only, real-time)')]:
    for i in range(n_states):
        ax.plot(common_idx, probs[:, i], lw=0.9, color=regime_colors[i], label=f'Regime {i}')
    shade_regimes(ax, common_idx, hmm_labels, regime_colors, alpha=0.15)
    shade_recessions(ax, recession)
    ax.set_title(title + ' — background = Viterbi path'); ax.set_ylabel('P(state | data)'); ax.grid(True, alpha=0.3)
axes[0].legend(loc='upper right', ncol=n_states)
plt.tight_layout(); plt.show()

## 5. Transition Matrix, Durations and Emission Means

In [ ]:
T = best_model.transmat_[np.ix_(state_order, state_order)]
T_df = pd.DataFrame(T, index=[f'Regime {i}' for i in range(n_states)], columns=[f'Regime {j}' for j in range(n_states)])
print('Transition matrix (row = from, column = to):')
print(T_df.round(4).to_string())

durations = [1 / (1 - T[i, i]) if T[i, i] < 1 else np.inf for i in range(n_states)]
eigenvalues, eigenvectors = np.linalg.eig(T.T)
stationary = np.real(eigenvectors[:, np.isclose(eigenvalues, 1.0)][:, 0])
stationary = stationary / stationary.sum()
print('\nExpected duration (months) / stationary probability:')
for i in range(n_states):
    print(f'  Regime {i}: {durations[i]:6.1f} months / {stationary[i]:.1%}')

emission_means = pd.DataFrame(best_model.means_[state_order], index=[f'Regime {i}' for i in range(n_states)], columns=pv_cols)
print('\nEmission means (proxy-variable space, 1 = pushes yields up):')
print(emission_means.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 4), gridspec_kw={'width_ratios': [1, 2.5]})
sns.heatmap(T_df, annot=True, fmt='.3f', cmap='Blues', vmin=0, vmax=1, ax=axes[0]); axes[0].set_title('Transition matrix')
sns.heatmap(emission_means.T, annot=True, fmt='.2f', cmap='RdYlGn', center=0.5, vmin=0, vmax=1, ax=axes[1]); axes[1].set_title('Emission means')
plt.tight_layout(); plt.show()

## 6. Recession Overlap and Agreement With the Bayesian Tree

In [ ]:
print('Recession overlap (Viterbi labels):')
for label in range(n_states):
    mask = hmm_labels == label
    overlap = (mask & rec_mask).sum()
    print(f'  Regime {label} ({mask.sum()} months): {overlap} recession months '
          f'({overlap / mask.sum():.1%} of regime, {overlap / max(rec_mask.sum(), 1):.1%} of all recessions)')

print(f'\nAgreement with the Bayesian tree (canonical labels):')
print(f'  accuracy (label-by-label): {(hmm_labels == regime_labels_canonical).mean():.1%}')
print(f'  Adjusted Rand Index:       {adjusted_rand_score(regime_labels_canonical, hmm_labels):.3f}')
print(f'  Normalized Mutual Info:    {normalized_mutual_info_score(regime_labels_canonical, hmm_labels):.3f}')
print(f'  ARI, filtered real-time labels vs tree: {adjusted_rand_score(regime_labels_canonical, hmm_labels_filtered):.3f}')

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(confusion_matrix(regime_labels_canonical, hmm_labels), annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'Regime {i}' for i in range(n_states)], yticklabels=[f'Regime {i}' for i in range(3)], ax=ax)
ax.set_xlabel('HMM (Viterbi)'); ax.set_ylabel('Bayesian Tree'); plt.tight_layout(); plt.show()

## 7. Macro-Only HMM (proxy variables without bond-market variables)
The main HMM sees yield information twice (through the Bond_Market proxy and through SHAP weights that target the yield
curve), so high agreement with the tree is expected. This fit uses the `xbond` proxy variables only.

In [ ]:
X_xbond = data_pv_xbond.values
print(f'Fitting macro-only HMM on {X_xbond.shape[1]} proxy variables: {data_pv_xbond.columns.tolist()}')
xbond_model, xbond_fits = fit_hmm(X_xbond, n_states, n_restarts)
xbond_labels, xbond_smoothed, xbond_filtered, xbond_order = decode(xbond_model, X_xbond, common_idx)
print(f'Best log-likelihood: {xbond_fits[0][0]:,.2f}')

print('\nRecession overlap (macro-only HMM):')
for label in range(n_states):
    mask = xbond_labels == label
    overlap = (mask & rec_mask).sum()
    print(f'  Regime {label} ({mask.sum()} months): {overlap} recession months ({overlap / max(rec_mask.sum(), 1):.1%} of all recessions)')

print('\nAgreement (Adjusted Rand Index):')
print(f'  macro-only HMM vs Bayesian tree: {adjusted_rand_score(regime_labels_canonical, xbond_labels):.3f}')
print(f'  macro-only HMM vs full HMM:      {adjusted_rand_score(hmm_labels, xbond_labels):.3f}')
print(f'  full HMM vs Bayesian tree:       {adjusted_rand_score(regime_labels_canonical, hmm_labels):.3f}')

In [ ]:
# Yield curve shaded by tree, full HMM and macro-only HMM regimes
plot_maturities = {'Y3M': '3M', 'Y6M': '6M', 'Y12M': '1Y', 'Y24M': '2Y', 'Y60M': '5Y', 'Y120M': '10Y'}
line_colors = ['#4575b4', '#74add1', '#abd9e9', '#fdae61', '#f46d43', '#d73027']
panels = [('Bayesian Tree (canonical)', regime_labels_canonical), ('Gaussian HMM — Viterbi', hmm_labels),
          ('Gaussian HMM — filtered (real-time) argmax', hmm_labels_filtered), ('Macro-only HMM — Viterbi', xbond_labels)]

fig, axes = plt.subplots(len(panels), 1, figsize=(16, 4.5 * len(panels)))
for ax, (title, labels) in zip(axes, panels):
    for (col, label), lc in zip(plot_maturities.items(), line_colors):
        ax.plot(common_idx, yields_svensson[col], color=lc, lw=1.2, label=label, zorder=3)
    shade_regimes(ax, common_idx, labels, regime_colors)
    shade_recessions(ax, recession)
    add_regime_legend(ax, labels, regime_colors)
    ax.set_title(title, fontweight='bold'); ax.set_ylabel('Yield (%)'); ax.grid(True, alpha=0.25)
    ax.set_xlim(common_idx.min() - pd.DateOffset(months=1), common_idx.max() + pd.DateOffset(months=1))
plt.tight_layout(); plt.show()

In [ ]:
# Every variable (levels) shaded by HMM regime
fig, axes = plt.subplots(len(all_cols), 1, figsize=(14, 3 * len(all_cols)))
for ax, col in zip(axes, all_cols):
    ax.plot(common_idx, data.loc[common_idx, col], color='black', lw=0.5, zorder=2)
    shade_regimes(ax, common_idx, hmm_labels, regime_colors)
    shade_recessions(ax, recession)
    ax.set_xlim(common_idx.min() - pd.DateOffset(months=1), common_idx.max() + pd.DateOffset(months=1))
    ax.set_title(f'{col} with HMM regime shading'); ax.set_ylabel(col)
add_regime_legend(axes[0], hmm_labels, regime_colors)
plt.suptitle('Macro Variables by HMM Regime', y=1.0)
plt.tight_layout(); plt.show()

## 8. Save

In [ ]:
log_lik, n_par, aic, bic = information_criteria(best_model, X_hmm, cov_type)

hmm_ts = pd.DataFrame({'regime': hmm_labels, 'regime_filtered': hmm_labels_filtered}, index=common_idx)
for i in range(n_states):
    hmm_ts[f'prob_smoothed_{i}'] = hmm_smoothed[:, i]
for i in range(n_states):
    hmm_ts[f'prob_filtered_{i}'] = hmm_filtered[:, i]
hmm_ts.to_csv(f'{data_folder}/MacroRegimes3_HMM_TimeSeries.csv', index_label='date')

xbond_ts = pd.DataFrame({'regime': xbond_labels, 'regime_filtered': xbond_filtered.argmax(axis=1)}, index=common_idx)
for i in range(n_states):
    xbond_ts[f'prob_smoothed_{i}'] = xbond_smoothed[:, i]
for i in range(n_states):
    xbond_ts[f'prob_filtered_{i}'] = xbond_filtered[:, i]
xbond_ts.to_csv(f'{data_folder}/MacroRegimes3_HMM_xbond_TimeSeries.csv', index_label='date')

T_df.to_csv(f'{data_folder}/MacroRegimes3_HMM_TransitionMatrix.csv')
emission_means.to_csv(f'{data_folder}/MacroRegimes3_HMM_EmissionMeans.csv')
model_selection.to_csv(f'{data_folder}/MacroRegimes3_HMM_ModelSelection.csv')

spread = (yields_svensson['Y120M'] - yields_svensson['Y24M']).values
regime_summary = pd.DataFrame({
    'n_months': [(hmm_labels == i).sum() for i in range(n_states)],
    'pct': [(hmm_labels == i).mean() for i in range(n_states)],
    'mean_spread': [spread[hmm_labels == i].mean() for i in range(n_states)],
    'expected_duration': durations,
    'stationary_prob': stationary,
    'recession_months': [((hmm_labels == i) & rec_mask).sum() for i in range(n_states)],
}, index=[f'Regime {i}' for i in range(n_states)])
regime_summary.to_csv(f'{data_folder}/MacroRegimes3_HMM_RegimeSummary.csv')

pd.DataFrame([{'n_states': n_states, 'cov_type': cov_type, 'min_covar': min_covar, 'n_obs': len(X_hmm), 'n_params': n_par,
               'log_likelihood': log_lik, 'converged': best_model.monitor_.converged, 'aic': aic, 'bic': bic,
               'top5_restart_mean_ari': ari[~np.eye(top_k, dtype=bool)].mean(),
               'ari_vs_tree': adjusted_rand_score(regime_labels_canonical, hmm_labels),
               'accuracy_vs_tree': (hmm_labels == regime_labels_canonical).mean(),
               'ari_xbond_vs_tree': adjusted_rand_score(regime_labels_canonical, xbond_labels)}]
             ).to_csv(f'{data_folder}/MacroRegimes3_HMM_ModelStats.csv', index=False)
print('Saved all MacroRegimes3_HMM_*.csv files.')